In [ ]:
import concurrent.futures
import numpy as np
import cv2  # OpenCV
from pymupdf import Document

def _check_page_for_rings(page_details):
    """
    Helper function.
    """
    page, dpi = page_details
    try:
        pix = page.get_pixmap(dpi=dpi, alpha=False)
        img_bytes = pix.tobytes("png")
        np_arr = np.frombuffer(img_bytes, np.uint8)
        img_cv_color = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)

        if img_cv_color is None:
            return False

        gray = cv2.cvtColor(img_cv_color, cv2.COLOR_BGR2GRAY) # Convert to grayscale

        thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                       cv2.THRESH_BINARY_INV, 11, 2) # Use the reliable adaptive thresholding

        contours, _ = cv2.findContours(thresh, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

        for cnt in contours:
            if not (100 < cv2.contourArea(cnt) < 20000):
                continue

            peri = cv2.arcLength(cnt, True)
            approx = cv2.approxPolyDP(cnt, 0.04 * peri, True)

            if len(approx) in [5, 6]:
                return True
    except Exception:
        return False

    return False

def pdf_contains_rings(pdf_path, dpi=150):
    try:
        doc = Document(pdf_path)
    except Exception:
        return False

    if doc.page_count == 0:
        return False

    found = False
    with concurrent.futures.ThreadPoolExecutor() as executor:
        page_details_list = [(doc.load_page(i), dpi) for i in range(doc.page_count)]

        futures = {executor.submit(_check_page_for_rings, p) for p in page_details_list}

        for future in concurrent.futures.as_completed(futures):
            if future.result():
                found = True
                executor.shutdown(wait=False, cancel_futures=True)
                break

    doc.close()
    return found

In [ ]:
import re
import io
import concurrent.futures
from pymupdf import Document
from PIL import Image
import pytesseract

def ocr_page(page_details):
    """
    Render a PyMuPDF page to an image and OCR it via Tesseract.
    Accepts a tuple (page_index, page, dpi) and returns the index and text.
    """
    page_index, page, dpi = page_details
    pix = page.get_pixmap(dpi=dpi, alpha=False)
    img = Image.open(io.BytesIO(pix.tobytes("png")))
    text = pytesseract.image_to_string(img)
    return page_index, text

def extract_claim_pages(pdf_path, dpi=150):
    """
    Locate pages containing the "Claims" section in a scanned PDF using parallel OCR.
    """
    doc = Document(pdf_path)
    if doc.page_count == 0:
        return [], ""

    start_patterns = [
        # Pattern 1: Matches lines starting with the word "claims", optionally
        # followed by a colon or a hyphen. This is the most common header for a claims section.
        # - ^ asserts position at the start of a line.
        # - \s* matches any whitespace character (zero or more times).
        # - [:-]? optionally matches a single character that is either a ":" or "-".
        #
        # Examples it will match:
        # "claims"
        # "CLAIMS:"
        # "Claims -"
        # "claims :"
        r"^claims\s*[:\-]?",

        # Pattern 2: Matches the common legal phrase "what is claimed is" or "what are claimed is".
        # This is a formal preamble to the claims in many patents.
        # - \b is a word boundary to ensure we match whole words.
        # - (?:is|are) matches either "is" or "are".
        # - (?:ed)? makes the "ed" in "claimed" optional, so it matches "claim" or "claimed".
        #
        # Examples it will match:
        # "What is claimed is"
        # "what are claim is"
        # "And now, what is claimed is:"
        r"\bwhat\s+(?:is|are)\s+claim(?:ed)?\s+is\b",

        # Pattern 3: A variation of the above, specifically matching the plural "claims".
        #
        # Examples it will match:
        # "What is claims is"
        # "what are claims is"
        r"\bwhat\s+(?:is|are)\s+claims\s+is\b",

        # Pattern 4: Matches another common preamble: "the invention claimed is".
        #
        # Examples it will match:
        # "The invention claimed is"
        # "the invention claim is"
        r"\bthe\s+invention\s+claim(?:ed)?\s+is\b"
    ]
    compiled_patterns = [re.compile(p, re.IGNORECASE | re.MULTILINE) for p in start_patterns]
    ocr_texts = [""] * doc.page_count

    # PHASE 1: Perform OCR on all pages in parallel to populate ocr_texts
    with concurrent.futures.ThreadPoolExecutor() as executor:
        page_details_list = [(i, doc.load_page(i), dpi) for i in range(doc.page_count)]
        futures = {executor.submit(ocr_page, p): p[0] for p in page_details_list}

        for future in concurrent.futures.as_completed(futures):
            page_index, text = future.result()
            ocr_texts[page_index] = text
            # The flawed logic of setting start_page here is now removed.

    doc.close()

    # PHASE 2: Sequentially search the results to find the *first* page that matches
    start_page = -1
    for i, text in enumerate(ocr_texts):
        if any(pat.search(text.lower()) for pat in compiled_patterns):
            start_page = i
            break  # Found the first one, stop looking

    if start_page == -1:
        return [], ""

    claim_pages_indices = list(range(start_page, len(ocr_texts)))
    claims_text = "\n".join(ocr_texts[start_page:])
    return claim_pages_indices, claims_text

In [58]:
def text_hints_markush(claims_text):
    """
    Return True if text contains R-group placeholders and one of several
    definition patterns indicating a Markush structure.
    """
    # Common R-group placeholders: R, R1, R2, etc.
    placeholder_patterns = [
        r"\bR\d*\b",                  # R or R1, R12...
        r"\bR-group\b",               # "R-group"
        r"\bR group\b",               # "R group"
        r"\bR\(\d+\)\b",              # R(1), R(2) in some specs
    ]

    # Definition patterns: different ways to express Markush lists or ranges
    definition_patterns = [
        r"selected from the group consisting(?: essentially of)?",  # standard
        r"selected from the group comprising",                      # comprising variant
        r"selected from(?: a group)? of",                           # shorter form
        r"wherein\s+R\d?",                                          # wherein R1 is …
        r"wherein said substituents\s+are",                         # alternative wording
        r"\bR\d?\s*to\s*R\d?\b",                                    # R1 to R5
        r"each R\d?\s*is\s*(?:a\s*)?[A-Za-z0-9,\- ]+",              # each R is …
        r"group consisting essentially of",                         # essential variant
    ]

    # Check for any placeholder
    has_placeholder = any(
        re.search(pat, claims_text, flags=re.IGNORECASE)
        for pat in placeholder_patterns
    )

    # Check for any definition phrase
    has_definition = any(
        re.search(pat, claims_text, flags=re.IGNORECASE)
        for pat in definition_patterns
    )

    return bool(has_placeholder and has_definition)

In [57]:
def save_claims_pdf(pdf_path, pages, out_path):
    """
    Extracts the specified pages from pdf_path and writes them to out_path.
    """
    if not pages:
        return
    with Document(pdf_path) as src, Document() as dst:
        dst.insert_pdf(src, from_page=min(pages), to_page=max(pages))
        dst.save(out_path)

In [69]:
from pathlib import Path

"""
Detects Markush structures in the "Claims" section of a patent PDF.
If found, extracts the Claims pages and:
  1) Prints the claims text.
  2) Writes the Claims pages out as a separate PDF: <original_name>_claims.pdf
Uses a hybrid CV+text approach.
"""

home = Path.home()
pdf_path = home / 'projects/Markush/data/2024/demo_202401012/2024-01/11/858/878/11858878.pdf'
patent_num = pdf_path.stem

print(f"📄 Processing patent: {patent_num}")

# 1. Fast CV pre-screening
print("🔬 Performing fast CV scan for chemical structures...")
if not pdf_contains_rings(pdf_path):
    print("❌ No potential Markush structures found by CV scan. Skipping detailed OCR.")
else:
    print("✅ Potential Markush structure found! Proceeding with full OCR of claims.")
    
    # 2. Extract claims using parallel OCR
    pages, claims_text = extract_claim_pages(pdf_path)

    if not pages:
        print("⚠️ Could not locate a Claims section, even though a structure was detected.")
    else:
        # 3. Final check with text heuristics and save
        if text_hints_markush(claims_text):
            print("✅ Markush-related text confirmed in Claims.")
            out_pdf = home / f'projects/Markush/data/2024/demo_202401012/{patent_num}_claims.pdf'
            save_claims_pdf(pdf_path, pages, out_pdf)
            print(f"🔖 Claims pages saved to: {out_pdf}")
        else:
            print("⚠️ A structure was found, but claims text did not confirm Markush definition.")


📄 Processing patent: 11858878
🔬 Performing fast CV scan for chemical structures...
✅ Potential Markush structure found! Proceeding with full OCR of claims.
✅ Markush-related text confirmed in Claims.
🔖 Claims pages saved to: /home/zqlyu2/projects/Markush/data/2024/demo_202401012/11858878_claims.pdf
